# Fair Code — Audit 05: Benefits Denial — Welfare Eligibility Bias

> *An automated means-test flags male applicants as ineligible at 18 percentage points higher than female applicants — not because of what they earn, but because of who they're married to.*

**Dataset:** UCI Adult Census Income — `adult.csv` (48,842 records)  
**Protected attributes:** Sex, Race, National Origin, Age  
**Proxy variables:** `relationship` (Husband/Wife encodes sex), `marital.status` (spousal rate reconstructs sex), `hours.per.week` (caregiving gap encodes sex), `occupation` (occupational segregation encodes race)  
**Fairness metric:** Demographic Parity  
**Model:** Random Forest Classifier  

---

Pipeline:
1. Load and explore the dataset
2. Identify the proxy variables (four of them)
3. Train the biased model (protected attributes + proxies included)
4. Measure the fairness gaps across four dimensions
5. Train the fair model (all removed)
6. Compare results

## 1. Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from sklearn.preprocessing import LabelEncoder
from scipy.stats import chi2_contingency

# Consistent styling across all Fair Code notebooks
plt.rcParams.update({
    'figure.facecolor': '#0d0f14',
    'axes.facecolor': '#131620',
    'axes.edgecolor': '#1e2130',
    'axes.labelcolor': '#b0aec0',
    'xtick.color': '#b0aec0',
    'ytick.color': '#b0aec0',
    'text.color': '#d4cfc0',
    'grid.color': '#1e2130',
    'grid.linestyle': '--',
    'font.family': 'monospace',
    'figure.dpi': 120
})

ACCENT = '#c9a84c'   # Fair Code gold
DANGER = '#9b2335'   # red — bias
SAFE   = '#4a7c6f'   # teal — mitigated
MUTED  = '#b0aec0'

print('Libraries loaded.')

## 2. Load and Explore the Dataset

In [ ]:
df_raw = pd.read_csv('Benefits Denial/adult.csv')
print(f'Dataset: {df_raw.shape[0]:,} rows, {df_raw.shape[1]} columns')
print(f'\nColumns: {list(df_raw.columns)}')
df_raw.head(3)

In [ ]:
df = df_raw.copy()

# Target: 1 = income >50K (flagged ineligible for benefits)
#         0 = income <=50K (eligible)
df['target']     = (df['income'] == '>50K').astype(int)
df['is_female']  = (df['sex'] == 'Female').astype(int)
df['is_foreign'] = (df['native.country'] != 'United-States').astype(int)
df['is_elderly'] = (df['age'] >= 55).astype(int)
df['is_minority'] = (~df['race'].isin(['White', 'Asian-Pac-Islander'])).astype(int)

print('Protected group breakdown:')
print(f'  Female applicants : {df["is_female"].mean():.1%}')
print(f'  Foreign-born      : {df["is_foreign"].mean():.1%}')
print(f'  Elderly (55+)     : {df["is_elderly"].mean():.1%}')
print(f'  Racial minorities : {df["is_minority"].mean():.1%}')

print(f'\nOverall ineligibility flag rate (income >50K): {df["target"].mean():.1%}')

print('\nIneligibility flag rate by sex (raw data):')
sex_raw = df.groupby('sex')['target'].mean() * 100
print(f'  Male  : {sex_raw["Male"]:.1f}%')
print(f'  Female: {sex_raw["Female"]:.1f}%')
print(f'  Raw gap: {sex_raw["Male"] - sex_raw["Female"]:.2f} percentage points')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 3.5))

# Ineligibility flag rate by sex
labels = ['Male', 'Female']
vals   = [sex_raw['Male'], sex_raw['Female']]
colors = [MUTED, DANGER]
bars = axes[0].bar(labels, vals, color=colors, width=0.4)
for bar, val in zip(bars, vals):
    axes[0].text(bar.get_x() + bar.get_width()/2, val + 0.3,
                 f'{val:.1f}%', ha='center', color=ACCENT, fontsize=11)
axes[0].set_title('Ineligibility flag rate by sex (raw data)', color=MUTED, fontsize=10)
axes[0].set_ylabel('% flagged ineligible')
axes[0].set_ylim(0, 50)
axes[0].grid(axis='y', alpha=0.3)

# Ineligibility flag rate by race
race_raw = df.groupby('is_minority')['target'].mean() * 100
labels2 = ['White/Asian-PI', 'Other minorities']
vals2   = [race_raw[0], race_raw[1]]
bars2 = axes[1].bar(labels2, vals2, color=[MUTED, DANGER], width=0.4)
for bar, val in zip(bars2, vals2):
    axes[1].text(bar.get_x() + bar.get_width()/2, val + 0.3,
                 f'{val:.1f}%', ha='center', color=ACCENT, fontsize=11)
axes[1].set_title('Ineligibility flag rate by race (raw data)', color=MUTED, fontsize=10)
axes[1].set_ylabel('% flagged ineligible')
axes[1].set_ylim(0, 50)
axes[1].grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()
print('Note: the raw gap exists before any model is trained — it is in the data itself.')

## 3. Identify the Proxy Variables

A proxy variable correlates with a protected attribute strongly enough to smuggle the bias back through the model — even after the protected column is removed.

This audit has **four** proxy variables:

| Proxy | Protected attribute | Mechanism |
|---|---|---|
| `relationship` | Sex | `Husband` is 0% female; `Wife` is 0% male. Sex with a different label. |
| `marital.status` | Sex | Male applicants are `Married-civ-spouse` at ~4x the female rate. |
| `hours.per.week` | Sex | Women average 6 fewer hours/week — a caregiving gap, not a productivity gap. |
| `occupation` | Race | Racial occupational segregation means job titles carry race signal. |

In [ ]:
def check_proxy(df, feature, protected_col):
    """Chi-squared test of independence. p < 0.05 = likely proxy."""
    contingency = pd.crosstab(df[feature], df[protected_col])
    chi2, p, dof, _ = chi2_contingency(contingency)
    return {'feature': feature, 'protected_attr': protected_col,
            'p_value': round(p, 6), 'is_proxy': p < 0.05}

# Test categorical proxies against sex
for feat in ['relationship', 'marital.status', 'occupation']:
    r = check_proxy(df, feat, 'sex')
    print(f"{r['feature']:<20} p={r['p_value']:<12} proxy={r['is_proxy']}")

# Test hours.per.week against sex (continuous — use correlation)
hrs_corr = df[['hours.per.week', 'is_female']].corr().iloc[0, 1]
print(f'hours.per.week       Pearson r={hrs_corr:.4f} (negative = women work fewer hours)')

In [ ]:
# relationship x sex
print('relationship distribution by sex (column proportions):')
rel_sex = pd.crosstab(df['relationship'], df['sex'], normalize='columns').round(3)
print(rel_sex.to_string())
print('  → Husband is 0% female. Wife is 0% male. This feature IS sex.')

# marital.status x sex
print('\nMarried-civ-spouse rate by sex:')
married = df.groupby('sex').apply(
    lambda x: (x['marital.status'] == 'Married-civ-spouse').mean()
).round(3)
print(f'  Female: {married["Female"]:.1%}')
print(f'  Male:   {married["Male"]:.1%}')

# hours.per.week x sex
print('\nhours.per.week mean by sex:')
hrs = df.groupby('sex')['hours.per.week'].mean().round(1)
print(f'  Female: {hrs["Female"]} hrs/wk')
print(f'  Male:   {hrs["Male"]} hrs/wk')

# occupation x race
print('\nHigh-skill occupation rate by race:')
df['high_occ'] = df['occupation'].isin(['Prof-specialty', 'Exec-managerial']).astype(int)
occ_race = df.groupby('race')['high_occ'].mean().round(3)
for r, v in occ_race.items():
    print(f'  {r:<25} {v:.1%}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
fig.suptitle('Proxy variable analysis — what the model learns without being told', color=ACCENT, fontsize=12, y=1.02)

# relationship distribution by sex
rel_plot = pd.crosstab(df['relationship'], df['sex'], normalize='columns') * 100
x = np.arange(len(rel_plot.index))
width = 0.35
axes[0].bar(x - width/2, rel_plot['Female'], width, color=DANGER, label='Female', alpha=0.85)
axes[0].bar(x + width/2, rel_plot['Male'],   width, color=MUTED,  label='Male',   alpha=0.85)
axes[0].set_xticks(x)
axes[0].set_xticklabels(rel_plot.index, rotation=20, ha='right')
axes[0].set_ylabel('Share of group (%)')
axes[0].set_title('relationship by sex — near-perfect sex proxy', color=MUTED, fontsize=10)
axes[0].legend()
axes[0].grid(axis='y', alpha=0.3)

# High-skill occupation by race
occ_plot = df.groupby('race')['high_occ'].mean().sort_values() * 100
bar_colors = [DANGER if r not in ['White', 'Asian-Pac-Islander'] else MUTED
              for r in occ_plot.index]
axes[1].barh(occ_plot.index, occ_plot.values, color=bar_colors, alpha=0.85)
axes[1].set_xlabel('% in Prof-specialty or Exec-managerial')
axes[1].set_title('Occupational segregation by race — race proxy', color=MUTED, fontsize=10)
axes[1].grid(axis='x', alpha=0.3)

plt.tight_layout()
plt.show()

## 4. Train the Biased Model

Features include `sex`, `race`, `age`, `native.country` directly **and** all four proxy variables.

In [ ]:
# Encode categoricals
cat_cols = [
    'workclass', 'education', 'marital.status', 'occupation',
    'relationship', 'race', 'sex', 'native.country'
]
df_enc = df.copy()
le = LabelEncoder()
for col in cat_cols:
    df_enc[col] = le.fit_transform(df_enc[col].astype(str))

print('Categorical columns encoded.')

In [ ]:
biased_features = [
    'age',              # protected attribute
    'workclass',
    'fnlwgt',
    'education',
    'education.num',
    'marital.status',   # proxy: encodes sex via spousal status
    'occupation',       # proxy: encodes race via occupational segregation
    'relationship',     # proxy: near-perfect sex encoding
    'race',             # protected attribute
    'sex',              # protected attribute
    'capital.gain',
    'capital.loss',
    'hours.per.week',   # proxy: encodes sex via caregiving gap
    'native.country',   # protected attribute (national origin)
]

X_biased = df_enc[biased_features]
y = df_enc['target']

X_train, X_test, y_train, y_test = train_test_split(
    X_biased, y, test_size=0.2, random_state=42
)

biased_model = RandomForestClassifier(n_estimators=100, random_state=42)
biased_model.fit(X_train, y_train)
biased_preds    = biased_model.predict(X_test)
biased_accuracy = accuracy_score(y_test, biased_preds)

results_b = X_test.copy()
results_b['pred']        = biased_preds
results_b['is_female']   = df.loc[X_test.index, 'is_female'].values
results_b['is_foreign']  = df.loc[X_test.index, 'is_foreign'].values
results_b['is_elderly']  = df.loc[X_test.index, 'is_elderly'].values
results_b['is_minority'] = df.loc[X_test.index, 'is_minority'].values

sex_b  = results_b.groupby('is_female')['pred'].mean() * 100
race_b = results_b.groupby('is_minority')['pred'].mean() * 100
geo_b  = results_b.groupby('is_foreign')['pred'].mean() * 100
age_b  = results_b.groupby('is_elderly')['pred'].mean() * 100

print('--- BIASED MODEL RESULTS ---')
print()
print(f'Male applicants    : {sex_b[0]:.2f}% flagged ineligible')
print(f'Female applicants  : {sex_b[1]:.2f}% flagged ineligible')
print(f'Fairness Gap (Sex) : {sex_b[0] - sex_b[1]:.2f}%')
print()
print(f'White/Asian-PI     : {race_b[0]:.2f}% flagged ineligible')
print(f'Other minorities   : {race_b[1]:.2f}% flagged ineligible')
print(f'Fairness Gap (Race): {race_b[0] - race_b[1]:.2f}%')
print()
print(f'US-born              : {geo_b[0]:.2f}% flagged ineligible')
print(f'Foreign-born         : {geo_b[1]:.2f}% flagged ineligible')
print(f'Fairness Gap (Origin): {geo_b[0] - geo_b[1]:.2f}%')
print()
print(f'Model Accuracy: {biased_accuracy:.2%}')

## 5. Train the Fair Model

All four protected attributes **and** all four proxy variables are removed. Only policy-defined economic signals remain.

In [ ]:
fair_features = [
    # age            removed (protected attribute)
    'workclass',       # retained: employment sector
    # fnlwgt         removed (census weight artefact)
    'education',       # retained: human capital signal
    'education.num',   # retained: same signal, numeric form
    # marital.status removed (proxy: encodes sex via spousal status)
    # occupation     removed (proxy: encodes race via occupational segregation)
    # relationship   removed (proxy: near-perfect sex encoding)
    # race           removed (protected attribute)
    # sex            removed (protected attribute)
    'capital.gain',    # retained: financial asset signal
    'capital.loss',    # retained: financial asset signal
    # hours.per.week removed (proxy: encodes sex via caregiving gap)
    # native.country removed (protected attribute: national origin)
]

X_fair = df_enc[fair_features]

X_train_f, X_test_f, y_train_f, y_test_f = train_test_split(
    X_fair, y, test_size=0.2, random_state=42
)

fair_model = RandomForestClassifier(n_estimators=100, random_state=42)
fair_model.fit(X_train_f, y_train_f)
fair_preds    = fair_model.predict(X_test_f)
fair_accuracy = accuracy_score(y_test_f, fair_preds)

results_f = X_test_f.copy()
results_f['pred']        = fair_preds
results_f['is_female']   = df.loc[X_test_f.index, 'is_female'].values
results_f['is_foreign']  = df.loc[X_test_f.index, 'is_foreign'].values
results_f['is_elderly']  = df.loc[X_test_f.index, 'is_elderly'].values
results_f['is_minority'] = df.loc[X_test_f.index, 'is_minority'].values

sex_f  = results_f.groupby('is_female')['pred'].mean() * 100
race_f = results_f.groupby('is_minority')['pred'].mean() * 100
geo_f  = results_f.groupby('is_foreign')['pred'].mean() * 100
age_f  = results_f.groupby('is_elderly')['pred'].mean() * 100

print('--- MITIGATED (UNBIASED) RESULTS ---')
print()
print(f'Male applicants          : {sex_f[0]:.2f}% flagged ineligible')
print(f'Female applicants        : {sex_f[1]:.2f}% flagged ineligible')
print(f'New Fairness Gap (Sex)   : {sex_f[0] - sex_f[1]:.2f}%')
print()
print(f'White/Asian-PI           : {race_f[0]:.2f}% flagged ineligible')
print(f'Other minorities         : {race_f[1]:.2f}% flagged ineligible')
print(f'New Fairness Gap (Race)  : {race_f[0] - race_f[1]:.2f}%')
print()
print(f'US-born                    : {geo_f[0]:.2f}% flagged ineligible')
print(f'Foreign-born               : {geo_f[1]:.2f}% flagged ineligible')
print(f'New Fairness Gap (Origin)  : {geo_f[0] - geo_f[1]:.2f}%')
print()
print(f'Model Accuracy: {fair_accuracy:.2%}')

## 6. Compare Results

In [ ]:
gap_sex_b  = sex_b[0]  - sex_b[1]
gap_sex_f  = sex_f[0]  - sex_f[1]
gap_race_b = race_b[0] - race_b[1]
gap_race_f = race_f[0] - race_f[1]
gap_geo_b  = geo_b[0]  - geo_b[1]
gap_geo_f  = geo_f[0]  - geo_f[1]

red_sex  = (gap_sex_b  - gap_sex_f)  / gap_sex_b  * 100
red_race = (gap_race_b - gap_race_f) / gap_race_b * 100
red_geo  = (gap_geo_b  - gap_geo_f)  / gap_geo_b  * 100

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
fig.suptitle('Benefits Denial — Biased vs Mitigated Model', color=ACCENT, fontsize=13, y=1.02)

dims = [
    (axes[0], [sex_b[0], sex_b[1]],   [sex_f[0], sex_f[1]],
     ['Male', 'Female'],               gap_sex_b,  gap_sex_f,  'Sex'),
    (axes[1], [race_b[0], race_b[1]], [race_f[0], race_f[1]],
     ['White/Asian-PI', 'Minorities'], gap_race_b, gap_race_f, 'Race'),
    (axes[2], [geo_b[0], geo_b[1]],   [geo_f[0], geo_f[1]],
     ['US-born', 'Foreign-born'],      gap_geo_b,  gap_geo_f,  'Origin'),
]

for ax, vals_b, vals_f, groups, gb, gf, label in dims:
    x = np.arange(len(groups))
    width = 0.35
    bars_b = ax.bar(x - width/2, vals_b, width, color=DANGER, label='Biased',    alpha=0.85)
    bars_f = ax.bar(x + width/2, vals_f, width, color=SAFE,   label='Mitigated', alpha=0.85)
    for bar, val in list(zip(bars_b, vals_b)) + list(zip(bars_f, vals_f)):
        ax.text(bar.get_x() + bar.get_width()/2, val + 0.3,
                f'{val:.1f}%', ha='center', fontsize=9, color=ACCENT)
    ax.set_xticks(x)
    ax.set_xticklabels(groups, fontsize=9)
    ax.set_ylim(0, 35)
    ax.set_ylabel('% flagged ineligible')
    ax.set_title(f'{label}: {gb:.2f}% → {gf:.2f}%', color=MUTED, fontsize=10)
    ax.legend(fontsize=8)
    ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

print(f'\nSummary')
print(f'-------')
print(f'Sex gap    before: {gap_sex_b:.2f}%   after: {gap_sex_f:.2f}%   reduction: {red_sex:.1f}%')
print(f'Race gap   before: {gap_race_b:.2f}%  after: {gap_race_f:.2f}%   reduction: {red_race:.1f}%')
print(f'Origin gap before: {gap_geo_b:.2f}%   after: {gap_geo_f:.2f}%   reduction: {red_geo:.1f}%')

## Key Insight

Removing `sex`, `race`, `age`, and `native.country` alone is not enough. Four proxy variables reconstruct the protected-class signal even after the direct columns are gone. `relationship` is the most egregious: `Husband` is assigned to 0% of female applicants and `Wife` to 0% of male applicants — it is sex encoded as a family role. `marital.status` reconstructs sex through spousal patterns. `hours.per.week` encodes the caregiving gap between men and women, not a difference in economic need. And `occupation` carries racial signal through decades of occupational segregation in the US labour market.

**The fix:** Remove the protected attributes **and** all four proxies. Retain only what a means-tested benefits programme can legitimately consult under equality law: education level, employment sector, and capital assets — the features that reflect financial need rather than who the applicant is.

---

*Part of the [Fair Code project](https://github.com/yakew7/Fair-Code) by [@thefaircodeproject](https://instagram.com/thefaircodeproject)*